# Try the Speaker LoRA adapter

Loads the pilot adapter (from `training/qlora_train.ipynb`) on top of the same 4-bit base and lets you
generate interactively. Colab/T4-friendly. For LOCAL use on the RTX 3060 Ti, see the notes at the bottom.

**Get the adapter onto the runtime first:** in VS Code, right-click `speaker_lora.zip` → **Upload to Colab**
(or, on the same runtime that trained it, `./speaker_lora` already exists).

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers peft trl triton
    !pip install --no-deps unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
import os, glob, zipfile

# Use ./speaker_lora if present (same runtime as training); else unzip an uploaded speaker_lora.zip.
ADAPTER = "speaker_lora"
if not os.path.isdir(ADAPTER):
    z = (glob.glob("speaker_lora.zip") or glob.glob("/content/**/speaker_lora.zip", recursive=True)
         or glob.glob("**/speaker_lora.zip", recursive=True))
    assert z, "Upload speaker_lora.zip to the runtime (VS Code: right-click -> Upload to Colab), then re-run."
    with zipfile.ZipFile(z[0]) as f:
        f.extractall(".")
    # the zip contains the 'speaker_lora' folder
    ADAPTER = "speaker_lora" if os.path.isdir("speaker_lora") else os.path.dirname(z[0])
assert os.path.exists(os.path.join(ADAPTER, "adapter_config.json")), f"no adapter in {ADAPTER}"
print("adapter:", os.path.abspath(ADAPTER))

In [ ]:
from unsloth import FastLanguageModel

# Unsloth reads the adapter's config, loads the 4-bit base it was trained on, and attaches the adapter.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER,        # local LoRA dir -> base + adapter
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("loaded.")

In [ ]:
def speaker(prompt, n=200, temp=0.9, top_p=0.9, rep=1.15):
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=n, do_sample=True, temperature=temp,
                         top_p=top_p, repetition_penalty=rep)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

# It's a BASE/continuation model — feed the START of a sentence in his register, not an instruction.
# Priming a casual/rant tone pulls out more of his voice than a neutral opener.
speaker("Abi ya, \u015fimdi bu konuya girince \u00e7\u0131ld\u0131r\u0131yorum. ")
print("\n" + "=" * 80 + "\n")
speaker("O\u011flum bak, bu dizinin olay\u0131 \u015fu: ")

In [ ]:
# Compare adapter-on vs adapter-off (the BASE model) on the same prompt — how much did the LoRA shift it?
p = "Bug\u00fcn \u015fu konuyu konu\u015faca\u011f\u0131z: "
print(">>> ADAPTER ON\n"); speaker(p)
with model.disable_adapter():
    print("\n>>> BASE (adapter off)\n")
    speaker(p)

## Running it on your local RTX 3060 Ti (8 GB)

The 4-bit base + this adapter fits in 8 GB. Two local options:

**A) GGUF + Ollama (smoothest on Windows — no Python/CUDA fuss).** On Colab, merge once:
```python
model.save_pretrained_merged("speaker_merged_16bit", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("speaker_gguf", tokenizer, quantization_method="q4_k_m")
```
Download the `.gguf` (~5 GB), then locally: `ollama create speaker -f Modelfile` (Modelfile = `FROM ./speaker.gguf`) and `ollama run speaker`.

**B) peft + transformers + bitsandbytes (Python).** `pip install transformers peft bitsandbytes accelerate`, then:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("unsloth/Meta-Llama-3.1-8B-bnb-4bit", device_map="auto")
model = PeftModel.from_pretrained(base, "speaker_lora")
tok = AutoTokenizer.from_pretrained("speaker_lora")
```
(bitsandbytes has Windows wheels now; unsloth itself is painful on Windows, so prefer plain peft locally.)

**Prompt format:** this is a transcript-continuation base model, so seed it with the *start* of a line in
his register (casual, direct-address) — not an instruction. The retrieval/`{Name}:` scaffold from CLAUDE.md
is for later, once the voice is stronger.